In [37]:
import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import joblib

Set paths

In [64]:
# Paths
train_dir = "/Users/saniabhandari/Documents/signlang/data2/asl_alphabet_train"
test_dir  = "/Users/saniabhandari/Documents/signlang/data2/asl_alphabet_test"

Load classes

In [62]:
classes = [c for c in os.listdir(train_dir) if not c.startswith('.')]
print("Number of classes:", len(classes))
print("Example classes:", classes[:5])

Number of classes: 29
Example classes: ['R', 'U', 'I', 'N', 'G']


Load Training Images

In [42]:
IMG_SIZE = 28  # Size for resizing images

X_train, y_train = [], []

for idx, cls in enumerate(classes[:26]):  # Only A-Z
    cls_path = os.path.join(train_dir, cls)
    for img_file in os.listdir(cls_path):
        if img_file.lower().endswith((".jpg", ".png")):
            img_path = os.path.join(cls_path, img_file)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  # Grayscale
            if img is None:
                continue
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))      # Resize
            X_train.append(img)
            y_train.append(idx)

X_train = np.array(X_train)
y_train = np.array(y_train)

print("Training images shape:", X_train.shape)
print("Training labels shape:", y_train.shape)


Training images shape: (78000, 28, 28)
Training labels shape: (78000,)


Load Test Images

In [65]:
test_classes = [c for c in os.listdir(test_dir) if not c.startswith('.')]
X_test, y_test = [], []

for idx, cls in enumerate(test_classes[:26]):  # Only A-Z
    cls_path = os.path.join(test_dir, cls)
    for img_file in os.listdir(cls_path):
        if img_file.lower().endswith((".jpg", ".png")):
            img_path = os.path.join(cls_path, img_file)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X_test.append(img)
            y_test.append(idx)

X_test = np.array(X_test)
y_test = np.array(y_test)

print("Test images shape:", X_test.shape)
print("Test labels shape:", y_test.shape)

Test images shape: (28, 28, 28)
Test labels shape: (28,)


Compute HOG Features

In [66]:
hog_params = {'orientations':12, 'pixels_per_cell':(4,4), 'cells_per_block':(2,2)}

def compute_hog_features_batch(images, hog_params):
    features = []
    for img in images:
        hog_vec = hog(img, **hog_params, block_norm='L2-Hys')
        features.append(hog_vec)
    return np.array(features)

X_train_hog = compute_hog_features_batch(X_train, hog_params)
X_test_hog = compute_hog_features_batch(X_test, hog_params)

print("HOG feature shape (train):", X_train_hog.shape)
print("HOG feature shape (test):", X_test_hog.shape)


HOG feature shape (train): (78000, 1728)
HOG feature shape (test): (28, 1728)


Feature scaling

In [67]:
scaler = StandardScaler()
X_train_hog_scaled = scaler.fit_transform(X_train_hog)
X_test_hog_scaled = scaler.transform(X_test_hog)

Train Random Forest

In [46]:
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=30,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_hog_scaled, y_train)

,n_estimators,500
,criterion,'gini'
,max_depth,30
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


Evaluate on separate test set

In [68]:
y_pred = model.predict(X_test_hog_scaled)
acc = accuracy_score(y_test, y_pred)
print("Test accuracy on separate test set:", acc*100, "%")

Test accuracy on separate test set: 3.571428571428571 %


/opt/miniconda3/envs/signlang/lib/python3.10/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")


Save the Model

In [49]:
model_data = {
    'model': model,
    'hog_params': hog_params,
    'scaler': scaler  # optional, keep for real-time inference
}

os.makedirs("../../models", exist_ok=True)
joblib.dump(model_data, "../../models/sign_model_asl.pkl")
print("Model saved to '../../models/sign_model_asl.pkl'")

Model saved to '../../models/sign_model_asl.pkl'
